# decting LoRA × DetectRL 全量（Colab 版）
有 GPU 自动全量训练+18 设置出分；无 GPU 自动冒烟。数据/代码来自 git 仓库。

In [ ]:
import os
REPO = 'https://github.com/woldy/raid.git'
if not os.path.exists('/content/decting-colab'):
    !git clone -q -b colab --depth 1 {REPO} /content/decting-colab
    !mkdir -p /content/data /content/work
    !tar xzf /content/decting-colab/detectrl.tar.gz -C /content/data
!ls /content/data/detectrl | head; du -sh /content/data

In [ ]:
#!/usr/bin/env python3
"""decting LoRA v1: Qwen3-0.6B + LoRA 序列分类, DetectRL 官方口径 (18 设置 test F1@0.5).

对照基线: TF-IDF 0.8930 | embedding+LR 0.9034 | Rob-Base 93.02 | Binoculars 79.61
Kaggle: GPU T4x2, internet ON (拉 HF 模型), 数据来自私有 dataset <OWNER>/decting-detectrl
"""
import json, os, random, sys, tarfile, time
from collections import Counter
from pathlib import Path

import numpy as np

SEED = 42
N_TRAIN = 6000          # v1: 采样版对齐 embedding 基线; v2 换全量
MAX_LEN = 512
BATCH = 16
GRAD_ACC = 2
EPOCHS = 2
LR = 1e-4
DATA_DIR = Path("/content/data/detectrl")
WORK = Path("/content/work")
random.seed(SEED); np.random.seed(SEED)


def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


def untar():
    # 数据集为散文件树（/kaggle/input/decting-detectrl/detectrl/**/*.json），
    # 无 tar.gz；直接 rglob 读取（2026-10-03 修复 PR 首跑 FileNotFoundError）
    out = {}
    for fp in sorted(DATA_DIR.rglob("*.json")):
        s = fp.read_text(encoding="utf-8", errors="ignore").strip()
        if not s:
            continue
        rows = json.loads(s) if s.startswith("[") else [json.loads(l) for l in s.splitlines() if l.strip()]
        out[str(fp.relative_to(DATA_DIR))] = rows
    log(f"loaded {len(out)} json files from dataset (loose files)")
    return out


def load_all():
    files = untar()
    trains, tests = {}, {}
    for name, rows in files.items():
        p = Path(name)
        if p.stem.endswith("_train"):
            trains[p.stem] = rows
        elif p.stem.endswith("_test"):
            setting = p.parent.name + "/" + p.stem.replace("_test", "")
            tests[setting] = rows
    log(f"files: {len(trains)} train, {len(tests)} test settings")
    return trains, tests


def wc(t):
    return len(t.split())


def build_train(trains):
    seen, rows = set(), []
    for name, rs in trains.items():
        for r in rs:
            t = r["text"]
            if wc(t) < 120:            # 对齐 test 过滤口径
                continue
            key = hash(t[:200])
            if key in seen:
                continue
            seen.add(key)
            rows.append((t, 1 if r["label"] == "llm" else 0))
    humans = [r for r in rows if r[1] == 0]
    llms = [r for r in rows if r[1] == 1]
    random.shuffle(llms)
    n_h = min(len(humans), N_TRAIN // 4)          # 1:3 平衡采样
    n_l = N_TRAIN - n_h
    sel = humans[:n_h] + llms[:n_l]
    random.shuffle(sel)
    log(f"train pool {len(rows)} (hu {len(humans)}/llm {len(llms)}), sampled {len(sel)} "
        f"({Counter(l for _, l in sel)})")
    return sel


def main():
    from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                              Trainer, TrainingArguments)
    from peft import LoraConfig, get_peft_model
    import torch
    from sklearn.metrics import f1_score

    # --- 兼容修复（2026-10-03 冒烟首跑 ERROR 的根因）---
    # Kaggle py3.13 镜像预装 torchao 0.10.0；新版 peft 的 LoRA dispatcher 在
    # dispatch_torchao→is_torchao_available() 检测到已安装但版本 <0.16.0 时直接抛
    # ImportError，不回退标准 Linear LoRA。本任务不用 torchao 量化 → 双层 patch 让
    # dispatcher 跳过 torchao 路径（is_torchao_available 按 False 处理）。
    import peft.import_utils as _piu
    _piu.is_torchao_available = lambda *_a, **_k: False
    try:
        import peft.tuners.lora.torchao as _lora_torchao
        _lora_torchao.is_torchao_available = lambda *_a, **_k: False
    except Exception:
        pass

    MODEL = "Qwen/Qwen3-0.6B"
    tok = AutoTokenizer.from_pretrained(MODEL)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    trains, tests = load_all()
    train_rows = build_train(trains)
    smoke = not torch.cuda.is_available()
    if smoke:
        log("[CPU SMOKE] 无 GPU——小样本全链验证（明天 GPU 配额重置后 push 同代码跑全量）")
        train_rows = train_rows[:32]
        tests = dict(list(sorted(tests.items()))[:1])
        for k in tests:
            tests[k] = tests[k][:32]

    # --- 兼容修复 3（v4 冒烟 OOM Killed）：CPU 内存有限 → 冒烟降载，GPU 全量口径不变 ---
    batch_size = 2 if smoke else BATCH
    grad_acc = 1 if smoke else GRAD_ACC
    max_len = 256 if smoke else MAX_LEN
    eval_batch = 8 if smoke else 64
    if smoke:
        log(f"[CPU SMOKE] 降载: batch={batch_size} grad_acc={grad_acc} max_len={max_len} eval_batch={eval_batch}")

    def encode(rows):
        texts = [t for t, _ in rows]
        labels = [l for _, l in rows]
        enc = tok(texts, truncation=True, max_length=max_len, padding=False)
        return [{"input_ids": i, "attention_mask": a, "labels": y}
                for i, a, y in zip(enc["input_ids"], enc["attention_mask"], labels)]

    class DS(torch.utils.data.Dataset):
        def __init__(self, items):
            self.items = items
        def __len__(self):
            return len(self.items)
        def __getitem__(self, i):
            return self.items[i]

    def collate(batch):
        mx = max(len(b["input_ids"]) for b in batch)
        pad = tok.pad_token_id
        ids, mask, labels = [], [], []
        for b in batch:
            d = mx - len(b["input_ids"])
            ids.append(b["input_ids"] + [pad] * d)
            mask.append(b["attention_mask"] + [0] * d)
            labels.append(b["labels"])
        return {"input_ids": torch.tensor(ids), "attention_mask": torch.tensor(mask),
                "labels": torch.tensor(labels)}

    # CPU 冒烟用 fp32（CPU 上 fp16 算子易崩）；GPU 侧 bf16 优先
    if torch.cuda.is_available():
        dt = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    else:
        dt = torch.float32
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2, torch_dtype=dt)
    model.config.pad_token_id = tok.pad_token_id
    lcfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                      target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
                      task_type="SEQ_CLS")
    model = get_peft_model(model, lcfg)
    model.print_trainable_parameters()
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    log(f"device={dev}, gpu={torch.cuda.get_device_name(0) if dev=='cuda' else '-'}")

    # --- 兼容修复 2（2026-10-03 v3 冒烟：TrainingArguments 不认 warmup_ratio）---
    # Kaggle 镜像 transformers 版本参数集与本地不同 → inspect 探测签名，
    # 仅传入该版本支持的参数，规避逐个 TypeError 试错浪费轮次。
    import inspect

    def make_args(**kw):
        sig = inspect.signature(TrainingArguments.__init__).parameters
        dropped = [k for k in kw if k not in sig]
        if dropped:
            log(f"[make_args] dropped unsupported kwargs: {dropped}")
        return TrainingArguments(**{k: v for k, v in kw.items() if k in sig})

    args = make_args(
        output_dir=str(WORK / "ck"), num_train_epochs=EPOCHS,
        max_steps=3 if smoke else -1,
        per_device_train_batch_size=batch_size, gradient_accumulation_steps=grad_acc,
        learning_rate=LR, warmup_ratio=0.05, logging_steps=25,
        save_strategy="no", report_to=[], fp16=(dev == "cuda" and not torch.cuda.is_bf16_supported()),
        bf16=(dev == "cuda" and torch.cuda.is_bf16_supported()),
        dataloader_num_workers=2, seed=SEED, remove_unused_columns=False)

    trainer = Trainer(model=model, args=args, train_dataset=DS(encode(train_rows)),
                      data_collator=collate)
    t0 = time.time()
    trainer.train()
    log(f"train done in {time.time()-t0:.0f}s")

    # ---- test 官方口径: 18 设置逐个 F1@0.5 ----
    model.eval()
    results = {}
    all_f1 = []
    for setting, rows in sorted(tests.items()):
        texts = [r["text"] for r in rows if wc(r["text"]) >= 120]
        labels = [1 if r["label"] == "llm" else 0 for r in rows if wc(r["text"]) >= 120]
        if not texts:
            continue
        probs = []
        B = eval_batch
        with torch.no_grad():
            for i in range(0, len(texts), B):
                enc = tok(texts[i:i + B], truncation=True, max_length=max_len,
                          padding=True, return_tensors="pt").to(dev)
                logits = model(**enc).logits
                probs.extend(torch.softmax(logits.float(), -1)[:, 1].cpu().tolist())
        preds = [1 if p >= 0.5 else 0 for p in probs]
        f1 = f1_score(labels, preds)
        results[setting] = {"n": len(texts), "f1@0.5": round(f1, 4),
                            "pos_rate": round(float(np.mean(labels)), 3)}
        all_f1.append(f1)
        log(f"  {setting}: F1={f1:.4f} (n={len(texts)})")

    avg = float(np.mean(all_f1))
    summary = {"model": "Qwen3-0.6B+LoRA(r16)", "n_train": len(train_rows), "epochs": EPOCHS,
               "smoke_mode": smoke,
               "avg_f1@0.5": round(avg, 4), "n_settings": len(all_f1),
               "baselines": {"tfidf": 0.8930, "embed_lr": 0.9034, "rob_base": 0.9302,
                             "binoculars": 0.7961}}
    log(f"AVG F1@0.5 = {avg:.4f} over {len(all_f1)} settings")
    (WORK / "results.json").write_text(json.dumps({"summary": summary, "per_setting": results},
                                                  indent=2, ensure_ascii=False))
    log("results.json saved")


if __name__ == "__main__":
    main()


In [ ]:
main()
# 结果展示
import json
print(json.dumps(json.load(open('/content/work/results.json'))['summary'], indent=1, ensure_ascii=False))